# 01 · Run the specialised text-to-CAD systems

One model per session on an **A100**. Predictions stream to Drive and every runner is
resumable, so a Colab timeout costs only the unfinished tail.

Set `MODEL` in the config cell and run top to bottom. Order below is cheapest first.

| MODEL | VRAM | ~time for 900 prompts |
|---|---|---|
| `text2cad` | <2 GB | ~25 min |
| `cadrille` / `cadrille-rl` | ~5 GB | ~40 min |
| `t2cq-qwen-3b` | ~8 GB | ~25 min |
| `cadmium-7b` | ~17 GB | ~50 min |
| `t2cq-mistral-7b` | ~16 GB | ~40 min |
| `cadfusion-v1.1` | ~18 GB | ~60 min |

In [ ]:
# Mount Drive so predictions survive a Colab timeout, then get the harness.
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/t2c_bench'
os.makedirs(WORK, exist_ok=True)
os.environ['T2C_WORK'] = WORK

BRANCH = 'main'   # set to a branch name to pick up work not yet merged
!git clone -q https://github.com/prashantkul366/T2C_Benchamrk /content/t2cbench_repo 2>/dev/null || true
%cd /content/t2cbench_repo
!git fetch -q origin && git checkout -q $BRANCH && git pull -q origin $BRANCH
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
print('work dir:', WORK)

In [ ]:
#@title Configuration
MODEL = 'cadmium-7b'  #@param ['text2cad','cadmium-7b','cadfusion-v1.1','cadrille','cadrille-rl','t2cq-qwen-3b','t2cq-mistral-7b']
SPLIT = 'A'           #@param ['A','B']
MODE  = 'pass_at_1'   #@param ['pass_at_1','best_of_k']
LIMIT = 8             #@param {type:'integer'}
# LIMIT = 8 is a SMOKE TEST: 8 prompts, ~1 min, and you get to look at the raw
# output before committing an hour of A100 time to a model whose weights or
# prompt format might be wrong. Set LIMIT = 0 for the full split once the
# smoke-test output looks like the right representation.

import yaml, os
cfg = yaml.safe_load(open('configs/models.yaml'))[MODEL]
SPLIT_FILE = f"{os.environ['T2C_WORK']}/data/split_{SPLIT.lower()}.jsonl"
suffix = f'_smoke{LIMIT}' if LIMIT else ''
OUT = f"{os.environ['T2C_WORK']}/results/raw/{MODEL}_split{SPLIT}_{MODE}{suffix}.jsonl"
LIM = f'--limit {LIMIT}' if LIMIT else ''
os.makedirs(os.path.dirname(OUT), exist_ok=True)
print(yaml.dump(cfg, sort_keys=False))
print('split :', SPLIT_FILE)
print('output:', OUT)
print('mode  :', 'SMOKE TEST' if LIMIT else 'FULL RUN')
# Smoke-test output goes to its own file so it never contaminates a full run.

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes sentencepiece
import torch; print(torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

### Gated weights

Two assets need a licence accepted on their HuggingFace page:

* **Text2CAD checkpoint** — `SadilKhan/Text2CAD` (needed only for `MODEL='text2cad'`)
* **Meta-Llama-3-8B** — the base CADFusion adapts (needed only for `cadfusion-v1.1`)

Everything else is open. Skip this cell for the other models.

In [ ]:
from huggingface_hub import login
login()   # paste a token with read access to the repos you accepted

### Run

In [ ]:
if cfg['runner'] == 'hf':
    base = f"--base {cfg['base_model']}" if cfg.get('lora') else ''
    sub  = f"--subfolder {cfg['subfolder']}" if cfg.get('subfolder') else ''
    chat = '' if cfg.get('chat_template', True) else '--no-chat-template'
    q4   = '--load-4bit' if cfg.get('load_4bit') else ''
    tmpl = cfg.get('template', 'general_one_shot')
    !python -m t2cbench.runners.run_hf \
        --model {cfg['weights']} {base} {sub} \
        --name {MODEL} --template {tmpl} {chat} {q4} {LIM} \
        --split {SPLIT_FILE} --out {OUT} --mode {MODE} --batch-size 8

elif cfg['runner'] == 'cadrille':
    !git clone -q --depth 1 https://github.com/col14m/cadrille /content/cadrille || true
    !pip install -q qwen-vl-utils
    n = 5 if MODE == 'best_of_k' else 1
    t = 0.7 if MODE == 'best_of_k' else 0.0
    !python -m t2cbench.runners.run_cadrille \
        --cadrille-repo /content/cadrille --checkpoint {cfg['weights']} \
        --name {MODEL} --split {SPLIT_FILE} --out {OUT} {LIM} \
        --n-samples {n} --temperature {t} --batch-size 16

elif cfg['runner'] == 'text2cad':
    !git clone -q --depth 1 https://github.com/SadilKhan/Text2CAD /content/Text2CAD || true
    from huggingface_hub import hf_hub_download
    ckpt = hf_hub_download(cfg['weights'], cfg['checkpoint_file'], repo_type='dataset')
    n = 5 if MODE == 'best_of_k' else 1
    !python -m t2cbench.runners.run_text2cad \
        --text2cad-repo /content/Text2CAD --checkpoint {ckpt} \
        --name {MODEL} --split {SPLIT_FILE} --out {OUT} --n-samples {n} {LIM}

### Check the output before you close the session

In [ ]:
import json
rows = [json.loads(l) for l in open(OUT)]
print(f'{len(rows)} generations, {len({r["sample_id"] for r in rows})} unique prompts')
print('\n--- first output ---')
print(rows[0]['output'][:900])

Now either switch `MODEL` and re-run, or move to notebook **03** to score everything.
Scoring is CPU-only, so do it in a separate CPU runtime.